<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Backend Module 2 (a): SQL Basics

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Learn

In this notebook, you will:

1. Say why a **database** exists at all, when a file looks so much simpler
2. Create a **table**, and map it onto the classes you already know
3. Put rows in with **INSERT** and get them out with **SELECT**
4. Ask sharper questions — **WHERE**, **ORDER BY**, **LIMIT**
5. Change and remove rows with **UPDATE** and **DELETE** — and the one habit that stops you wiping a table
6. Link two tables with a **FOREIGN KEY**, and watch the database refuse bad data
7. Pull them back together with a **JOIN**

> **Nothing to install, and no API key.** This runs on `sqlite3`, which ships inside Python — so the
> whole notebook works in Colab on any laptop.
>
> **Start the PostgreSQL download first.** In VS Code, run `docker compose up -d`, then come back
> here while it pulls. By the time you finish this notebook your database will be ready.

## 1. How to Use This Notebook

Run every cell top to bottom, in order — later cells use tables the earlier ones created.

**Don't just read it.** After a cell runs, change a value and run it again. Change `'CSE'` to `'ECE'`.
Change `20` to `21`. Break it on purpose and read the error. That habit is worth more than any
explanation here.

| Part | Sections | What it is |
|---|---|---|
| **A · One table** | 2–6 | create it, fill it, question it, change it |
| **B · Two tables** | 7–8 | foreign keys and JOIN — how real data is shaped |
| **C · The rest** | 9–10 | what exists beyond today, and how SQLite differs from Postgres |

---

# Part A — One Table

---

## 2. Why a Database, and Not Just a File?

Your Module-1 API kept students in a Python list. Stop the server and they're gone. The obvious fix
is "write it to a file" — so why doesn't anyone do that?

| The problem | What a file does | What a database does |
|---|---|---|
| Two requests save at once | one corrupted file | both handled, safely |
| Find the CSE students among 100,000 | read all 100,000 | goes straight to them |
| Power cut halfway through a write | half a file | all of it, or none of it |
| Someone saves a duplicate email | accepts it | **refuses it** |

> A database is not a fancy file. It's a program whose whole job is to be careful with your data
> while many people touch it at once.

Run this once — it sets up an empty database and a small helper that prints results as a table.

In [ ]:
import sqlite3

# A whole database, living in memory. Nothing is installed and no file is created.
con = sqlite3.connect(":memory:")
con.execute("PRAGMA foreign_keys = ON")     # section 7 needs this


def run(sql, params=()):
    """Run some SQL and print the result as a table."""
    cur = con.execute(sql, params)
    if cur.description is None:             # CREATE / INSERT / UPDATE / DELETE
        con.commit()
        print(f"OK — {cur.rowcount} row(s) affected" if cur.rowcount >= 0 else "OK")
        return
    headers = [d[0] for d in cur.description]
    rows = cur.fetchall()
    if not rows:
        print("(no rows)")
        return
    width = [max(len(str(r[i])) for r in rows + [tuple(headers)]) for i in range(len(headers))]
    print("  ".join(h.ljust(w) for h, w in zip(headers, width)))
    print("-" * (sum(width) + 2 * (len(width) - 1)))
    for r in rows:
        print("  ".join(str(v).ljust(w) for v, w in zip(r, width)))


print("ready")

## 3. Your First Table

A table is a **class**. A row is an **object**. A column is an **attribute**. You already know how to
model data — this is a second syntax for it.

| Python | SQL |
|---|---|
| `class Student:` | `CREATE TABLE students` |
| one object | one **row** |
| `self.name` | a **column** `name` |
| `age: int` | a column type `INTEGER` |
| `Field(min_length=2)` | a **constraint** (`NOT NULL`, `UNIQUE`) |

Three constraints do most of the work:

- **`PRIMARY KEY`** — identifies a row uniquely, forever
- **`NOT NULL`** — must have a value
- **`UNIQUE`** — no duplicates allowed

The difference from Pydantic: the **database** enforces these, so they hold even if someone
bypasses your API completely.

In [ ]:
# Table names are plural by convention: a table holds many students.
run("""
CREATE TABLE students (
    id      INTEGER PRIMARY KEY,
    name    TEXT    NOT NULL,
    branch  TEXT    NOT NULL DEFAULT 'CSE',
    age     INTEGER NOT NULL,
    email   TEXT    NOT NULL UNIQUE
)
""")

## 4. Putting Rows In — `INSERT`

You name the columns, then the values. `id` is left out on purpose — the database assigns it.

In [ ]:
run("INSERT INTO students (name, branch, age, email) VALUES ('Ada', 'CSE', 20, 'ada@lpu.in')")

In [ ]:
# Several at once. Change a name or an age here and re-run the whole section.
con.executemany(
    "INSERT INTO students (name, branch, age, email) VALUES (?, ?, ?, ?)",
    [("Raj",   "ECE", 21, "raj@lpu.in"),
     ("Meera", "CSE", 19, "meera@lpu.in"),
     ("Farah", "MECH", 22, "farah@lpu.in")],
)
con.commit()
print("inserted")

Now try inserting Ada's email a second time. The database **refuses** it — that's `UNIQUE` doing its
job, and it's the thing a file could never do for you.

In [ ]:
try:
    run("INSERT INTO students (name, branch, age, email) VALUES ('Fake Ada', 'CSE', 20, 'ada@lpu.in')")
except sqlite3.IntegrityError as e:
    print("refused:", e)

## 5. Getting Rows Out — `SELECT`

`SELECT` is the one you'll write most. `*` means every column.

In [ ]:
run("SELECT * FROM students")

In [ ]:
# Ask for only what you need. Fewer columns = less data over the network.
run("SELECT name, branch FROM students")

## 6. Sharper Questions — `WHERE`, `ORDER BY`, `LIMIT`

`WHERE` filters rows. This is where a database stops being a list.

In [ ]:
run("SELECT name, age FROM students WHERE branch = 'CSE'")

In [ ]:
# Change 'CSE' to 'cse' and run it again. Zero rows — string comparison is
# case-sensitive. Five seconds now saves an hour of confusion later.
run("SELECT name, age FROM students WHERE branch = 'cse'")

In [ ]:
# Comparisons, sorting, and a cap on how many come back.
run("SELECT name, age FROM students WHERE age >= 20 ORDER BY age DESC LIMIT 2")

## 7. Changing and Removing — `UPDATE`, `DELETE`

🚨 **The most important cell in this notebook.**

`UPDATE students SET branch = 'ECE';` — with no `WHERE` — changes **every row in the table**.
`DELETE FROM students;` empties it. There is no undo, and no confirmation.

> **The habit that protects you: write the `WHERE` first, then go back and write the `UPDATE`.**

Everyone eventually runs a `WHERE`-less update on something that matters. The habit is the only
thing standing between you and that afternoon.

In [ ]:
run("UPDATE students SET branch = 'ECE' WHERE name = 'Meera'")
run("SELECT name, branch FROM students")

In [ ]:
run("DELETE FROM students WHERE name = 'Farah'")
run("SELECT name, branch FROM students")

# Part B — Two Tables

---

## 8. Foreign Keys — One Course, Many Students

Real data is never one table. A **course** has many **students**; each student belongs to one course.

A **foreign key is a column that holds another table's primary key.** That is the whole definition.

💡 Remember Module 1's `self.courses = []` — *"a Student HAS courses"*? This is that same has-a
relationship, written down permanently. A database calls it a **relationship**.

In [ ]:
run("""
CREATE TABLE courses (
    code  TEXT PRIMARY KEY,
    title TEXT NOT NULL
)
""")

con.executemany("INSERT INTO courses (code, title) VALUES (?, ?)",
                [("CSE101", "Intro to Backend"), ("ECE201", "Signals")])
con.commit()
run("SELECT * FROM courses")

In [ ]:
# A second students table, this time with a link to courses.
run("""
CREATE TABLE enrolled (
    id          INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    course_code TEXT REFERENCES courses(code)      -- <- the foreign key
)
""")

con.executemany("INSERT INTO enrolled (name, course_code) VALUES (?, ?)",
                [("Ada", "CSE101"), ("Raj", "CSE101"), ("Meera", "ECE201")])
con.commit()
run("SELECT * FROM enrolled")

Now point a student at a course that doesn't exist. The database refuses — it will not let your data
become nonsense. **That refusal is the entire reason foreign keys exist.**

In [ ]:
try:
    run("INSERT INTO enrolled (name, course_code) VALUES ('Ghost', 'NOPE999')")
except sqlite3.IntegrityError as e:
    print("refused:", e)

## 9. `JOIN` — Putting Them Back Together

A JOIN glues two tables together on a matching column.

Read it out loud, in this order:
> *"Take `enrolled`, glue on the course whose `code` matches its `course_code`, then give me these
> two columns."*

Students who can narrate a JOIN can write one.

In [ ]:
run("""
SELECT enrolled.name, courses.title
FROM enrolled
JOIN courses ON enrolled.course_code = courses.code
""")

In [ ]:
# A JOIN is still just a query — WHERE works exactly the same on it.
run("""
SELECT enrolled.name, courses.title
FROM enrolled
JOIN courses ON enrolled.course_code = courses.code
WHERE courses.code = 'CSE101'
""")

# Part C — What's Next

---

## 10. Named, Not Taught 📖

These are real, useful, and not in today's three hours. Read them at home — you now know enough SQL
that each is a short hop.

| Idea | What it's for |
|---|---|
| **`GROUP BY` + `COUNT`/`AVG`** | "how many students per branch" in one query |
| **Indexes** | why `WHERE email = ...` on a million rows can be instant |
| **Transactions** (`BEGIN`/`COMMIT`) | "money left A" and "money reached B" both happen, or neither does |
| **Subqueries** | a `SELECT` inside a `SELECT` |
| **Migrations** (Alembic) | changing a table's shape after real data is already in it |

Here's `GROUP BY` once, so the shape isn't a total stranger:

In [ ]:
run("SELECT branch, COUNT(*) AS how_many FROM students GROUP BY branch")

## 11. SQLite vs PostgreSQL

You learned SQL on **SQLite** because it needs nothing installed. The rest of the module uses
**PostgreSQL**. The SQL you just wrote is about 95% portable — here's the 5% that isn't:

| | SQLite (this notebook) | PostgreSQL (the rest of the module) |
|---|---|---|
| Lives in | one file, or memory | a running server |
| Auto id | `INTEGER PRIMARY KEY` | `SERIAL` / `GENERATED … AS IDENTITY` |
| Types | suggestions, mostly | enforced |
| Concurrent writers | one | many |
| `VARCHAR(80)` limits | ignored | enforced |
| Foreign keys | **off** unless you turn them on | always on |

> And in the next session, SQLAlchemy hides even that 5% — you'll write Python, and it writes the
> right dialect for whichever database is behind it.

---

## 12. Exercises

Fill in the `___`. Run each cell to check yourself. The tables from above are still loaded.

### Q1. Every student in the `ECE` branch

**Hint:** `WHERE column = 'value'` — and remember it's case-sensitive.

In [ ]:
run("SELECT * FROM students WHERE ___ = '___'")

### Q2. The two oldest students, name and age only

**Hint:** you need `ORDER BY`, the direction `DESC`, and `LIMIT`.

In [ ]:
run("SELECT name, age FROM students ORDER BY ___ ___ LIMIT ___")

### Q3. Add yourself to the table

**Hint:** the columns are `name`, `branch`, `age`, `email` — and the email must be one nobody has used.

In [ ]:
run("INSERT INTO students (___, ___, ___, ___) VALUES ('___', '___', ___, '___')")
run("SELECT * FROM students")

### Q4. Move Raj into `MECH` — and *only* Raj

**Hint:** write the `WHERE` first. Then go back and write the `SET`.

In [ ]:
run("UPDATE students SET ___ = '___' WHERE ___ = '___'")
run("SELECT name, branch FROM students")

### Q5. Every enrolled student, with the **title** of their course

**Hint:** join `enrolled` to `courses`, matching `course_code` against `code`.

In [ ]:
run("""
SELECT enrolled.name, courses.___
FROM enrolled
JOIN courses ON enrolled.___ = courses.___
""")

### Q6. Everyone on `ECE201`, by name

**Hint:** it's Q5 plus one more line.

In [ ]:
run("""
SELECT enrolled.name
FROM enrolled
JOIN courses ON enrolled.course_code = courses.code
WHERE courses.___ = '___'
""")

---

## Key Takeaways

1. **A table is a class, a row is an object.** You already knew how to model data — this is a second syntax.
2. **Constraints are enforced by the database**, not by your API. `UNIQUE` holds even if someone bypasses your code entirely.
3. **`SELECT` … `WHERE` is the workhorse.** Filtering in the database beats fetching everything and filtering in Python.
4. **Write the `WHERE` first.** An `UPDATE` or `DELETE` without one hits every row, with no undo.
5. **A foreign key is just a column holding another table's key** — and it makes the database refuse data that doesn't add up.
6. **A `JOIN` reads as a sentence.** Take this table, glue on the matching row from that one, give me these columns.

> Next: the same tables, on a real PostgreSQL server — and then SQLAlchemy, which writes this SQL
> for you while you write Python.